In [ ]:
import numpy as np
from sklearn.datasets import make_moons

np.random.seed(15)

X, y_labels = make_moons(n_samples=200, noise=0.1, random_state=42)

# y_labels is (200,) with values 0 or 1
# Reshape to (200, 2) as one-hot for your 2-output network
y = np.zeros((200, 2))
y[np.arange(200), y_labels] = 1

W1 = np.random.randn(2, 2) * 0.1
b1 = np.zeros(2)
W2 = np.random.randn(2, 2) * 0.1
b2 = np.zeros(2)
N = len(X)

In [ ]:
def compute_loss(X, y, W1, b1, W2, b2):
    loss = 0
    for i in range(len(X)):
        z1 = W1 @ X[i] + b1
        a1 = np.maximum(z1, 0)
        z2 = W2 @ a1 + b2
        loss += np.sum((z2 - y[i])**2)
    return loss / N

# Gradient

In [ ]:
def compute_loss_partials(X, y, W1, b1, W2, b2):
    grad = np.zeros(12)

    for i in range(N):
        z1 = W1 @ X[i] + b1
        a1 = np.maximum(z1, 0)
        z2 = W2 @ a1 + b2

        dz1_2 = 2/N * (z2[0] - y[i][0])
        dz2_2 = 2/N * (z2[1] - y[i][1])

        dw11_2 = dz1_2 * a1[0]
        dw12_2 = dz1_2 * a1[1]

        dw21_2 = dz2_2 * a1[0]
        dw22_2 = dz2_2 * a1[1]

        db1_2 = dz1_2 * 1
        db2_2 = dz2_2 * 1

        da1_1 = dz1_2 * W2[0][0] + dz2_2 * W2[1][0]
        da2_1 = dz1_2 * W2[0][1] + dz2_2 * W2[1][1]

        dz1_1 = da1_1 * (z1[0] > 0)
        dz2_1 = da2_1 * (z1[1] > 0)

        dw11_1 = dz1_1 * X[i][0]
        dw12_1 = dz1_1 * X[i][1]
        
        dw21_1 = dz2_1 * X[i][0]
        dw22_1 = dz2_1 * X[i][1]

        db1_1 = dz1_1 * 1
        db2_1 = dz2_1 * 1

        grad += np.array([
            dw11_2,
            dw12_2,
            dw21_2,
            dw22_2,
            db1_2,
            db2_2,
            dw11_1,
            dw12_1,
            dw21_1,
            dw22_1,
            db1_1,
            db2_1,
        ])
    dW2 = grad[0:4].reshape((2, 2))
    db2 = grad[4:6]
    dW1 = grad[6:10].reshape((2, 2))
    db1 = grad[10:12]
    return dW2, db2, dW1, db1

In [ ]:
def numerical_gradient_check(X, y, W1, b1, W2, b2, epsilon=1e-8):
    """Compare analytical gradients to numerical gradients."""

    # Get your analytical gradients and flatten to match params layout
    dW2, db2_grad, dW1, db1_grad = compute_loss_partials(X, y, W1, b1, W2, b2)
    grad = np.concatenate([dW2.flatten(), db2_grad, dW1.flatten(), db1_grad])

    # Pack all parameters into one vector (same order as your grad)
    params = np.array([
        W2[0,0], W2[0,1], W2[1,0], W2[1,1],
        b2[0], b2[1],
        W1[0,0], W1[0,1], W1[1,0], W1[1,1],
        b1[0], b1[1],
    ])

    numerical_grad = np.zeros_like(params)

    for idx in range(len(params)):
        # Nudge parameter up
        params_plus = params.copy()
        params_plus[idx] += epsilon
        loss_plus = compute_loss_with_params(X, y, params_plus)

        # Nudge parameter down
        params_minus = params.copy()
        params_minus[idx] -= epsilon
        loss_minus = compute_loss_with_params(X, y, params_minus)

        # Numerical gradient: rise over run
        numerical_grad[idx] = (loss_plus - loss_minus) / (2 * epsilon)

    # Compare
    diff = np.abs(grad - numerical_grad)
    relative_diff = diff / (np.maximum(np.abs(grad), np.abs(numerical_grad)) + 1e-8)

    print("Max absolute diff:", np.max(diff))
    print("Max relative diff:", np.max(relative_diff))
    print("All close?", np.allclose(grad, numerical_grad, atol=1e-5))

    # Print side by side
    for i in range(len(params)):
        print(f"Param {i}: analytical={grad[i]:.8f}  numerical={numerical_grad[i]:.8f}  diff={diff[i]:.2e}")


def compute_loss_with_params(X, y, params):
    """Forward pass using flat parameter vector."""
    W2 = params[0:4].reshape(2, 2)
    b2 = params[4:6]
    W1 = params[6:10].reshape(2, 2)
    b1 = params[10:12]

    loss = compute_loss(X, y, W1, b1, W2, b2)
    return loss

In [ ]:
numerical_gradient_check(X, y, W1, b1, W2, b2)

# Stochastic Gradient Descent

In [ ]:
def sgd(X, y, W1, W2, b1, b2, lr=1e-3, n_iters=1000):
    W1 = W1.copy()
    W2 = W2.copy()
    b1 = b1.copy()
    b2 = b2.copy()
    losses = []

    for _ in range(n_iters):
        dW2, db2, dW1, db1 = compute_loss_partials(X, y, W1, b1, W2, b2)
        W1 -= lr * dW1
        W2 -= lr * dW2
        b1 -= lr * db1
        b2 -= lr * db2
        losses.append(compute_loss(X, y, W1, b1, W2, b2))

    return W1, W2, b1, b2, losses

print(f"Loss Before SGD: {compute_loss(X, y, W1, b1, W2, b2)}")
sgd_W1, sgd_W2, sgd_b1, sgd_b2, sgd_losses = sgd(X, y, W1, W2, b1, b2)
print(f"Loss After SGD: {compute_loss(X, y, sgd_W1, sgd_b1, sgd_W2, sgd_b2)}")

# Momentum

In [ ]:
def momentum(X, y, W1, W2, b1, b2, lr=1e-3, gamma=0.9, n_iters=1000):
    W1 = W1.copy()
    W2 = W2.copy()
    b1 = b1.copy()
    b2 = b2.copy()
    losses = []

    vW2, vb2, vW1, vb1 = np.zeros_like(W2), np.zeros_like(b2), np.zeros_like(W1), np.zeros_like(b1)
    for _ in range(n_iters):
        dW2, db2, dW1, db1 = compute_loss_partials(X, y, W1, b1, W2, b2)
        vW2 = gamma*vW2 + lr*dW2
        vb2 = gamma*vb2 + lr*db2
        vW1 = gamma*vW1 + lr*dW1
        vb1 = gamma*vb1 + lr*db1

        W1 -= vW1
        W2 -= vW2
        b1 -= vb1
        b2 -= vb2
        losses.append(compute_loss(X, y, W1, b1, W2, b2))

    return W1, W2, b1, b2, losses

print(f"Loss Before momentum: {compute_loss(X, y, W1, b1, W2, b2)}")
mom_W1, mom_W2, mom_b1, mom_b2, mom_losses = momentum(X, y, W1, W2, b1, b2)
print(f"Loss After momentum: {compute_loss(X, y, mom_W1, mom_b1, mom_W2, mom_b2)}")

# Adam

In [ ]:
def adam(X, y, W1, W2, b1, b2, beta_1=0.9, beta_2=0.999, lr=1e-3, eps=1e-8, n_iters=1000):
    W1 = W1.copy()
    W2 = W2.copy()
    b1 = b1.copy()
    b2 = b2.copy()
    losses = []

    mW2, mb2, mW1, mb1 = np.zeros_like(W2), np.zeros_like(b2), np.zeros_like(W1), np.zeros_like(b1)
    vW2, vb2, vW1, vb1 = np.zeros_like(W2), np.zeros_like(b2), np.zeros_like(W1), np.zeros_like(b1)
    for t in range(1, n_iters + 1):
        dW2, db2, dW1, db1 = compute_loss_partials(X, y, W1, b1, W2, b2)
        # First moment of gradient
        mW2 = beta_1*mW2 + (1-beta_1)*dW2
        mb2 = beta_1*mb2 + (1-beta_1)*db2
        mW1 = beta_1*mW1 + (1-beta_1)*dW1
        mb1 = beta_1*mb1 + (1-beta_1)*db1

        # Second moment of gradient
        vW2 = (beta_2*vW2 + (1-beta_2)*dW2**2)
        vb2 = (beta_2*vb2 + (1-beta_2)*db2**2)
        vW1 = (beta_2*vW1 + (1-beta_2)*dW1**2)
        vb1 = (beta_2*vb1 + (1-beta_2)*db1**2)

        W1 -= lr/((vW1/(1 - beta_2**t))**0.5 + eps) * mW1/(1 - beta_1**t)
        W2 -= lr/((vW2/(1 - beta_2**t))**0.5 + eps) * mW2/(1 - beta_1**t)
        b1 -= lr/((vb1/(1 - beta_2**t))**0.5 + eps) * mb1/(1 - beta_1**t)
        b2 -= lr/((vb2/(1 - beta_2**t))**0.5 + eps) * mb2/(1 - beta_1**t)
        losses.append(compute_loss(X, y, W1, b1, W2, b2))

    return W1, W2, b1, b2, losses

print(f"Loss Before Adam: {compute_loss(X, y, W1, b1, W2, b2)}")
adam_W1, adam_W2, adam_b1, adam_b2, adam_losses = adam(X, y, W1, W2, b1, b2)
print(f"Loss After Adam: {compute_loss(X, y, adam_W1, adam_b1, adam_W2, adam_b2)}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, scale in zip(axes, ["linear", "log"]):
    ax.plot(sgd_losses, label="SGD", linewidth=1.5)
    ax.plot(mom_losses, label="Momentum", linewidth=1.5)
    ax.plot(adam_losses, label="Adam", linewidth=1.5)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss")
    ax.set_yscale(scale)
    ax.set_title(f"Loss vs iteration ({scale} y-axis)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()